In [0]:
%sql
-- ============================================
-- Market Price Overview
-- საბაზრო ფასების benchmark ცხრილი
-- ============================================

CREATE OR REPLACE TABLE getdata.calculated.market_price_overview AS
SELECT 
    make,
    model,
    model_year,
    COUNT(*) AS listing_count,
    ROUND(AVG(price), 0) AS avg_market_price,
    ROUND(PERCENTILE(price, 0.5), 0) AS median_price,
    MIN(price) AS min_price,
    MAX(price) AS max_price,
    ROUND(AVG(mileage), 0) AS avg_mileage
FROM getdata.calculated.cleaned_used_cars
GROUP BY make, model, model_year
HAVING COUNT(*) >= 5;  -- საკმარისი sample რომ იყოს!

In [0]:
%sql
-- ============================================
-- Bargain Deals Detection
-- მანქანები რომლებიც 10%+ იაფია market ფასზე
-- ============================================

CREATE OR REPLACE TABLE getdata.calculated.market_bargain_deals AS
SELECT 
    Cars.id,
    Cars.make,
    Cars.model,
    Cars.model_year,
    Cars.price AS listing_price,
    Cars.mileage,
    Cars.condition,
    Cars.state,
    overview.avg_mileage,
    overview.avg_market_price,
    overview.median_price,
    ROUND((Cars.price - overview.median_price) * 100.0 / overview.median_price, 1) AS discount_percentage,
    ROUND(overview.median_price - Cars.price, 0) AS savings_amount
FROM getdata.calculated.cleaned_used_cars AS Cars
INNER JOIN getdata.calculated.market_price_overview AS overview
    ON overview.make = Cars.make
   AND overview.model = Cars.model
   AND overview.model_year = Cars.model_year
   AND overview.median_price >= 1000 -- outliers რომ გამოვრიცხოთ
WHERE Cars.price < overview.median_price * 0.9  -- 10%+ იაფია
  AND ROUND((Cars.price - overview.median_price) * 100.0 / overview.median_price, 1) > -40
  AND Cars.condition IN ('Excellent', 'Good', 'Like New')  -- მხოლოდ კარგი მდგომარეობის
ORDER BY discount_percentage ASC
LIMIT 1000; -- top 1000 deal

In [0]:
%sql
-- ============================================
-- State Market Summary
-- შტატების მიხედვით ბაზრის მიმოხილვა
-- ============================================

CREATE OR REPLACE TABLE getdata.calculated.market_state_summary AS
WITH StateData AS (
    SELECT 
        state,
        price,
        mileage,
        make,
        model,
        ROW_NUMBER() OVER (PARTITION BY state ORDER BY make, model) AS rn
    FROM getdata.calculated.cleaned_used_cars
)
SELECT 
    state,
    COUNT(*) AS total_listings,
    ROUND(AVG(price), 0) AS avg_price,
    ROUND(PERCENTILE(price, 0.5), 0) AS median_price,
    ROUND(AVG(mileage), 0) AS avg_mileage,
    FIRST(make) AS most_common_make,
    FIRST(model) AS most_common_model,
    COUNT(DISTINCT make) AS unique_makes
FROM StateData
GROUP BY state
HAVING COUNT(*) >= 50  -- მინიმუმ 50 listing შტატში
ORDER BY total_listings DESC;

In [0]:
%sql
-- ============================================
-- Depreciation Analysis
-- გაუფასურების ანალიზი წლების მიხედვით
-- ============================================

CREATE OR REPLACE TABLE getdata.calculated.market_depreciation_analysis AS
SELECT 
    make,
    model,
    model_year,
    ROUND(AVG(price), 0) AS avg_price,
    LAG(ROUND(AVG(price), 0), 1) OVER (
        PARTITION BY make, model 
        ORDER BY model_year
    ) AS prev_year_price,
    ROUND(
        (ROUND(AVG(price), 0) - LAG(ROUND(AVG(price), 0), 1) OVER (
            PARTITION BY make, model 
            ORDER BY model_year
        )) * 100.0 / LAG(ROUND(AVG(price), 0), 1) OVER (
            PARTITION BY make, model 
            ORDER BY model_year
        ),
        1
    ) AS depreciation_rate,
    COUNT(*) AS sample_size
FROM getdata.calculated.cleaned_used_cars
WHERE model_year >= 2015
GROUP BY make, model, model_year
HAVING COUNT(*) >= 15  -- საკმარისი sample
QUALIFY prev_year_price IS NOT NULL
ORDER BY make, model, model_year DESC;